# Business Analysis & KPI Generation

## Objective

Convert the Customer 360 dataset into business-level KPIs, segment performance metrics, actionable findings, and recommendations.

## Analysis Areas

* Overall banking KPIs
* Customer and financial metrics
* Segment-level performance
* Customer engagement
* Product usage
* Business findings
* Business recommendations
* Submission-ready outputs


In [1]:
import pandas as pd

customer_360 = pd.read_csv("../data/final/customer_360.csv")

print("Customer 360 loaded successfully")
print("Rows:", customer_360.shape[0])
print("Columns:", customer_360.shape[1])

Customer 360 loaded successfully
Rows: 5000
Columns: 29


In [2]:
print("Customer 360 columns:")
print("-" * 40)

for i, col in enumerate(customer_360.columns, start=1):
    print(f"{i}. {col}")

Customer 360 columns:
----------------------------------------
1. customer_id
2. age
3. gender
4. country
5. occupation
6. annual_income
7. credit_score
8. tenure_years
9. join_date
10. last_transaction_date
11. late_loan_count
12. default_loan_count
13. complaint_count
14. escalated_count
15. total_balance
16. active_account_count
17. recency_days
18. products_number
19. churn
20. loan_to_income_ratio
21. total_transactions
22. total_transaction_value
23. avg_transaction_value
24. loan_count
25. total_loan_amount
26. average_interest_rate
27. total_service_interactions
28. average_resolution_days
29. account_count


In [3]:
# Overall KPI
total_customers = customer_360["customer_id"].nunique()

total_deposits = customer_360["total_balance"].sum()

total_loan_amount = customer_360["total_loan_amount"].sum()

total_transactions = customer_360["total_transactions"].sum()

total_transaction_value = customer_360["total_transaction_value"].sum()

average_balance = customer_360["total_balance"].mean()

average_loan_amount = customer_360.loc[
    customer_360["loan_count"] > 0,
    "total_loan_amount"
].mean()

average_transaction_value = customer_360["avg_transaction_value"].mean()

loan_customers = (customer_360["loan_count"] > 0).sum()

active_customers = (customer_360["active_account_count"] > 0).sum()

loan_penetration = (loan_customers / total_customers) * 100

active_customer_rate = (active_customers / total_customers) * 100

print("OVERALL BANKING KPIs")
print("=" * 50)

print(f"Total Customers: {total_customers:,}")
print(f"Total Deposits / Balance: {total_deposits:,.2f}")
print(f"Total Loan Amount: {total_loan_amount:,.2f}")
print(f"Total Transactions: {total_transactions:,.0f}")
print(f"Total Transaction Value: {total_transaction_value:,.2f}")
print(f"Average Customer Balance: {average_balance:,.2f}")
print(f"Average Loan Amount: {average_loan_amount:,.2f}")
print(f"Average Transaction Value: {average_transaction_value:,.2f}")
print(f"Loan Customers: {loan_customers:,}")
print(f"Active Customers: {active_customers:,}")
print(f"Loan Penetration: {loan_penetration:.2f}%")
print(f"Active Customer Rate: {active_customer_rate:.2f}%")

OVERALL BANKING KPIs
Total Customers: 5,000
Total Deposits / Balance: 429,675,148.55
Total Loan Amount: 945,526,046.31
Total Transactions: 100,000
Total Transaction Value: 29,633,739.80
Average Customer Balance: 85,935.03
Average Loan Amount: 481,919.49
Average Transaction Value: 296.05
Loan Customers: 1,962
Active Customers: 3,644
Loan Penetration: 39.24%
Active Customer Rate: 72.88%


In [4]:
# Check whether the segmentation notebook saved a segmented file
import os

print("Files in data/final:")
print("-" * 40)

for file in os.listdir("../data/final"):
    print(file)

Files in data/final:
----------------------------------------
customer_360.csv
customer_360_segmented.csv
customer_360_sql.csv


In [5]:
# Load the segmented Customer 360
customer_360_segmented = pd.read_csv(
    "../data/final/customer_360_segmented.csv"
)

print("Segmented Customer 360 loaded successfully")
print("Rows:", customer_360_segmented.shape[0])
print("Columns:", customer_360_segmented.shape[1])

Segmented Customer 360 loaded successfully
Rows: 5000
Columns: 31


In [6]:
print("\nSegment columns:")
print("-" * 40)

print(
    customer_360_segmented[
        ["customer_id", "segment", "segment_name"]
    ].head()
)


Segment columns:
----------------------------------------
  customer_id  segment            segment_name
0      C00001        0  High-Balance Customers
1      C00002        1  Core Banking Customers
2      C00003        1  Core Banking Customers
3      C00004        1  Core Banking Customers
4      C00005        1  Core Banking Customers


In [7]:
print("\nSegment distribution:")
print("-" * 40)

print(
    customer_360_segmented["segment_name"].value_counts()
)


Segment distribution:
----------------------------------------
segment_name
Core Banking Customers       2218
High-Service Customers       1242
High-Balance Customers       1090
High-Value Loan Customers     450
Name: count, dtype: int64


In [8]:
# Create Segment-Level KPIs
segment_kpis = (
    customer_360_segmented
    .groupby("segment_name")
    .agg(
        customers=("customer_id", "nunique"),
        total_balance=("total_balance", "sum"),
        total_loan_amount=("total_loan_amount", "sum"),
        total_transactions=("total_transactions", "sum"),
        transaction_value=("total_transaction_value", "sum"),
        avg_balance=("total_balance", "mean"),
        avg_income=("annual_income", "mean"),
        avg_credit_score=("credit_score", "mean"),
        avg_products=("products_number", "mean"),
        avg_service_interactions=("total_service_interactions", "mean")
    )
    .reset_index()
)
segment_kpis.round(2)

,segment_name,customers,total_balance,total_loan_amount,total_transactions,transaction_value,avg_balance,avg_income,avg_credit_score,avg_products,avg_service_interactions
0,Core Banking Customers,2218,9.391089e+07,1.468653e+08,44231,13001685.61,42340.35,66009.60,652.46,1.25,1.16
1,High-Balance Customers,1090,2.329643e+08,1.311245e+08,21949,6525002.81,213728.69,68881.83,651.41,3.68,1.74
2,High-Service Customers,1242,6.748205e+07,1.025137e+08,24698,7403875.49,54333.37,65519.00,654.29,1.50,3.72
3,High-Value Loan Customers,450,3.531793e+07,5.650225e+08,9122,2703175.89,78484.29,62176.44,650.83,2.99,2.02


In [9]:
# Verify the KPI table
print("Segment KPI columns:")
print("-" * 40)

print(segment_kpis.columns.tolist())

print("\nShape:")
print(segment_kpis.shape)

print("\nKPI table:")
print(segment_kpis.to_string(index=False))

Segment KPI columns:
----------------------------------------
['segment_name', 'customers', 'total_balance', 'total_loan_amount', 'total_transactions', 'transaction_value', 'avg_balance', 'avg_income', 'avg_credit_score', 'avg_products', 'avg_service_interactions']

Shape:
(4, 11)

KPI table:
             segment_name  customers  total_balance  total_loan_amount  total_transactions  transaction_value   avg_balance   avg_income  avg_credit_score  avg_products  avg_service_interactions
   Core Banking Customers       2218    93910894.42       146865308.84               44231        13001685.61  42340.349152 66009.603246        652.463931      1.254734                  1.158251
   High-Balance Customers       1090   232964274.69       131124509.76               21949         6525002.81 213728.692376 68881.834862        651.413761      3.681651                  1.740367
   High-Service Customers       1242    67482049.97       102513727.53               24698         7403875.49  54333.3735

In [10]:
# Segment Performance
segment_kpis["customer_pct"] = (
    segment_kpis["customers"] / total_customers * 100
)

segment_kpis["balance_pct"] = (
    segment_kpis["total_balance"] / total_deposits * 100
)

segment_kpis["loan_pct"] = (
    segment_kpis["total_loan_amount"] / total_loan_amount * 100
)

segment_kpis["transaction_pct"] = (
    segment_kpis["transaction_value"] / total_transaction_value * 100
)

In [11]:
segment_kpis[
    [
        "segment_name",
        "customers",
        "customer_pct",
        "balance_pct",
        "loan_pct",
        "transaction_pct"
    ]
].round(2)

,segment_name,customers,customer_pct,balance_pct,loan_pct,transaction_pct
0,Core Banking Customers,2218,44.36,21.86,15.53,43.87
1,High-Balance Customers,1090,21.80,54.22,13.87,22.02
2,High-Service Customers,1242,24.84,15.71,10.84,24.98
3,High-Value Loan Customers,450,9.00,8.22,59.76,9.12


In [12]:
# Key Segment Findings
largest_segment = segment_kpis.loc[
    segment_kpis["customers"].idxmax(), "segment_name"
]

highest_balance_segment = segment_kpis.loc[
    segment_kpis["total_balance"].idxmax(), "segment_name"
]

highest_loan_segment = segment_kpis.loc[
    segment_kpis["total_loan_amount"].idxmax(), "segment_name"
]

highest_transaction_segment = segment_kpis.loc[
    segment_kpis["transaction_value"].idxmax(), "segment_name"
]

lowest_engagement_segment = segment_kpis.loc[
    segment_kpis["avg_products"].idxmin(), "segment_name"
]

In [13]:
print("KEY SEGMENT FINDINGS")
print("=" * 40)

print("Largest segment:", largest_segment)
print("Highest balance:", highest_balance_segment)
print("Highest loan exposure:", highest_loan_segment)
print("Highest transaction value:", highest_transaction_segment)
print("Lowest product engagement:", lowest_engagement_segment)

KEY SEGMENT FINDINGS
Largest segment: Core Banking Customers
Highest balance: High-Balance Customers
Highest loan exposure: High-Value Loan Customers
Highest transaction value: Core Banking Customers
Lowest product engagement: Core Banking Customers


In [14]:
# Create Business Insights
high_balance = segment_kpis[
    segment_kpis["segment_name"] == "High-Balance Customers"
].iloc[0]

high_loan = segment_kpis[
    segment_kpis["segment_name"] == "High-Value Loan Customers"
].iloc[0]

core = segment_kpis[
    segment_kpis["segment_name"] == "Core Banking Customers"
].iloc[0]

In [15]:
high_service = segment_kpis[
    segment_kpis["segment_name"] == "High-Service Customers"
].iloc[0]

print("BUSINESS INSIGHTS")
print("=" * 60)

print(
    f"1. Core Banking Customers are the largest segment, "
    f"representing {core['customer_pct']:.2f}% of customers."
)

print(
    f"2. High-Balance Customers represent {high_balance['customer_pct']:.2f}% "
    f"of customers but hold {high_balance['balance_pct']:.2f}% "
    f"of the total customer balance."
)

print(
    f"3. High-Value Loan Customers represent only "
    f"{high_loan['customer_pct']:.2f}% of customers but account for "
    f"{high_loan['loan_pct']:.2f}% of total loan exposure."
)

print(
    f"4. High-Service Customers represent {high_service['customer_pct']:.2f}% "
    f"of customers and generate {high_service['transaction_pct']:.2f}% "
    f"of total transaction value."
)

print(
    f"5. Core Banking Customers generate the highest transaction "
    f"value, contributing {core['transaction_pct']:.2f}% of total transaction value."
)

BUSINESS INSIGHTS
1. Core Banking Customers are the largest segment, representing 44.36% of customers.
2. High-Balance Customers represent 21.80% of customers but hold 54.22% of the total customer balance.
3. High-Value Loan Customers represent only 9.00% of customers but account for 59.76% of total loan exposure.
4. High-Service Customers represent 24.84% of customers and generate 24.98% of total transaction value.
5. Core Banking Customers generate the highest transaction value, contributing 43.87% of total transaction value.


In [16]:
# Business Recommendations
# Business Recommendations
print("BUSINESS RECOMMENDATIONS")
print("=" * 60)

print("1. Core Banking Customers")
print("   Largest segment with about 1.25 products per customer: focus on")
print("   cross-selling a second product and increasing product adoption.")

print("\n2. High-Balance Customers")
print("   Hold over half of all customer balance: protect with retention offers")
print("   and dedicated relationship-manager coverage.")

print("\n3. High-Value Loan Customers")
print("   Hold most of the loan exposure: prioritize credit monitoring,")
print("   repayment-status tracking, and exposure limits.")

print("\n4. High-Service Customers")
print("   Highest complaint volume and churn flag: fix complaint resolution first,")
print("   then offer targeted retention and digital self-service.")

BUSINESS RECOMMENDATIONS
1. Core Banking Customers
   Largest segment with about 1.25 products per customer: focus on
   cross-selling a second product and increasing product adoption.

2. High-Balance Customers
   Hold over half of all customer balance: protect with retention offers
   and dedicated relationship-manager coverage.

3. High-Value Loan Customers
   Hold most of the loan exposure: prioritize credit monitoring,
   repayment-status tracking, and exposure limits.

4. High-Service Customers
   Highest complaint volume and churn flag: fix complaint resolution first,
   then offer targeted retention and digital self-service.


In [17]:
# Find KPI Summary
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Total Customer Balance",
        "Total Loan Amount",
        "Total Transactions",
        "Total Transaction Value",
        "Average Customer Balance",
        "Average Loan Amount",
        "Average Transaction Value",
        "Loan Penetration",
        "Active Customer Rate"
    ],
    
    "Value": [
        total_customers,
        total_deposits,
        total_loan_amount,
        total_transactions,
        total_transaction_value,
        average_balance,
        average_loan_amount,
        average_transaction_value,
        loan_penetration,
        active_customer_rate
    ]
})

kpi_summary

,KPI,Value
0,Total Customers,5.000000e+03
1,Total Customer Balance,4.296751e+08
2,Total Loan Amount,9.455260e+08
3,Total Transactions,1.000000e+05
4,Total Transaction Value,2.963374e+07
5,Average Customer Balance,8.593503e+04
6,Average Loan Amount,4.819195e+05
7,Average Transaction Value,2.960547e+02
8,Loan Penetration,3.924000e+01
9,Active Customer Rate,7.288000e+01


In [18]:
# Format KPI Summary
kpi_summary["Value"] = kpi_summary["Value"].round(2)

kpi_summary

,KPI,Value
0,Total Customers,5.000000e+03
1,Total Customer Balance,4.296751e+08
2,Total Loan Amount,9.455260e+08
3,Total Transactions,1.000000e+05
4,Total Transaction Value,2.963374e+07
5,Average Customer Balance,8.593503e+04
6,Average Loan Amount,4.819195e+05
7,Average Transaction Value,2.960500e+02
8,Loan Penetration,3.924000e+01
9,Active Customer Rate,7.288000e+01


In [19]:
segment_summary = segment_kpis[
    [
        "segment_name",
        "customers",
        "customer_pct",
        "total_balance",
        "balance_pct",
        "total_loan_amount",
        "loan_pct",
        "transaction_value",
        "transaction_pct"
    ]
].copy()

segment_summary

,segment_name,customers,customer_pct,total_balance,balance_pct,total_loan_amount,loan_pct,transaction_value,transaction_pct
0,Core Banking Customers,2218,44.36,9.391089e+07,21.856255,1.468653e+08,15.532656,13001685.61,43.874603
1,High-Balance Customers,1090,21.80,2.329643e+08,54.218699,1.311245e+08,13.867890,6525002.81,22.018830
2,High-Service Customers,1242,24.84,6.748205e+07,15.705365,1.025137e+08,10.841978,7403875.49,24.984614
3,High-Value Loan Customers,450,9.00,3.531793e+07,8.219682,5.650225e+08,59.757476,2703175.89,9.121953


In [20]:
# Export Analysis Results
import os

os.makedirs("../outputs/exports", exist_ok=True)

print("Export folder ready")

Export folder ready


In [21]:
kpi_summary.to_csv(
    "../outputs/exports/kpi_summary.csv",
    index=False
)

segment_summary.to_csv(
    "../outputs/exports/segment_summary.csv",
    index=False
)

print("KPI summary and segment summary exported")

KPI summary and segment summary exported


In [22]:
with pd.ExcelWriter("../outputs/exports/business_analysis.xlsx") as writer:
    kpi_summary.to_excel(
        writer,
        sheet_name="KPI Summary",
        index=False
    )
    
    segment_summary.to_excel(
        writer,
        sheet_name="Segment Summary",
        index=False
    )

print("Excel report exported successfully")

Excel report exported successfully


In [23]:
# Final Validation
import os

export_files = os.listdir("../outputs/exports")

print("BUSINESS ANALYSIS EXPORTS")
print("=" * 40)

for file in export_files:
    print(file)

BUSINESS ANALYSIS EXPORTS
.ipynb_checkpoints
business_analysis.xlsx
dashboard_data.csv
dashboard_kpis.csv
kpi_summary.csv
segment_summary.csv
sql_channel_preference.csv
sql_kpis.csv
sql_monthly_trend.csv
sql_product_holding.csv


In [24]:
print("Business Analysis")
print("=" * 40)

print("Total Customers:", total_customers)
print("Loan Penetration:", f"{loan_penetration:.2f}%")
print("Active Customer Rate:", f"{active_customer_rate:.2f}%")
print("Segments:", customer_360_segmented["segment_name"].nunique())

print("\nKey Segments:")
for segment in segment_kpis["segment_name"]:
    print("-", segment)

print("\nExports:")
print("- kpi_summary.csv")
print("- segment_summary.csv")
print("- business_analysis.xlsx")

Business Analysis
Total Customers: 5000
Loan Penetration: 39.24%
Active Customer Rate: 72.88%
Segments: 4

Key Segments:
- Core Banking Customers
- High-Balance Customers
- High-Service Customers
- High-Value Loan Customers

Exports:
- kpi_summary.csv
- segment_summary.csv
- business_analysis.xlsx
